## Load Broyles EPD Data

In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('../02_processed_data/broyles_epd_data_cleaned.csv')
df.head()

,Company,Company Location - Street,Company Location - City,Company Location - State,Company Location - Zip,Plant,Plant Location - Street,Plant Location - City,Plant Location - State,Plant Location - Zip,...,A1-A3 Global Warming Potential (kg CO2-eq),A1 GWP,A2 GWP,A3 GWP,A1-A3 Global Warming Potential (kg CO2-eq)_per_CY,A1 GWP_per_CY,A2 GWP_per_CY,A3 GWP_per_CY,contains_fly_ash,contains_slag
0,American Rock Products (ARP),2580 Hagen Road,Richland,WA,99352,Boardman Plant,71427 E. Columbia Ave,Boardman,OR,97818,...,238.0,228.0,5.25,5.16,181.96,174.32,4.01,3.95,False,False
1,American Rock Products (ARP),2580 Hagen Road,Richland,WA,99352,Boardman Plant,71427 E. Columbia Ave,Boardman,OR,97818,...,302.0,287.0,10.00,5.16,230.90,219.43,7.65,3.95,True,False
2,American Rock Products (ARP),2580 Hagen Road,Richland,WA,99352,Hermiston Plant,81830 S. US Hwy 395,Hermiston,OR,97838,...,238.0,224.0,10.20,4.07,181.96,171.26,7.80,3.11,False,False
3,American Rock Products (ARP),2580 Hagen Road,Richland,WA,99352,Hermiston Plant,81830 S. US Hwy 395,Hermiston,OR,97838,...,320.0,284.0,31.40,4.07,244.66,217.13,24.01,3.11,True,False
4,American Rock Products (ARP),2580 Hagen Road,Richland,WA,99352,Pendleton Plant,73569 McKay Lane,Pendleton,OR,97801,...,353.0,306.0,28.60,18.70,269.89,233.95,21.87,14.30,True,False


In [8]:
# Round compressive strength to nearest 500 psi to match EC3 bucket structure
df['Compressive_Strength'] = ((df['Concrete Compressive Strength (psi)'] / 500).round() * 500).astype(int)

# Filter to 2500–8000 psi inclusive
df_filtered = df[
    (df['Compressive_Strength'] >= 2500) & (df['Compressive_Strength'] <= 8000)
].copy()

# Classify SCM type from boolean flags
def classify_scm(row):
    if row['contains_fly_ash'] and row['contains_slag']:
        return 'Both Fly Ash & Slag'
    elif row['contains_fly_ash']:
        return 'Fly Ash'
    elif row['contains_slag']:
        return 'Slag'
    else:
        return 'No SCM'

df_filtered['scm_type'] = df_filtered.apply(classify_scm, axis=1)
df_filtered['gwp_rounded'] = df_filtered['A1-A3 Global Warming Potential (kg CO2-eq)_per_CY'].round(0)

scm_plot_order = ['No SCM', 'Fly Ash', 'Slag', 'Both Fly Ash & Slag']
colors         = ['#2c7fb8', '#41b6c4', '#7fcdbb', '#c7e9b4']
color_map      = dict(zip(scm_plot_order, colors))

df_plot = df_filtered[df_filtered['scm_type'].isin(scm_plot_order)].copy()

print(f"Total samples (2500–8000 psi): {len(df_filtered)}")
print(f"\nSamples by SCM Type:")
print(df_filtered['scm_type'].value_counts())

Total samples (2500–8000 psi): 43110

Samples by SCM Type:
scm_type
Fly Ash                17572
No SCM                 14068
Slag                    8756
Both Fly Ash & Slag     2714
Name: count, dtype: int64


In [9]:
strength_order  = sorted(df_plot['Compressive_Strength'].unique())
strength_to_idx = {s: i for i, s in enumerate(strength_order)}

n_groups    = len(scm_plot_order)
group_width = 0.8
box_width   = group_width / n_groups
offsets_map = {
    scm: (i - (n_groups - 1) / 2) * box_width
    for i, scm in enumerate(scm_plot_order)
}

POINT_JITTER_WIDTH  = 0.04
MAX_DOTS_PER_BUCKET = 600

fig = go.Figure()
rng = np.random.default_rng(seed=42)

# --- Box traces ---
for scm_type, color in zip(scm_plot_order, colors):
    subset   = df_plot[df_plot['scm_type'] == scm_type]
    x_values = [strength_to_idx[s] + offsets_map[scm_type]
                for s in subset['Compressive_Strength']]

    fig.add_trace(go.Box(
        x=x_values,
        y=subset['gwp_rounded'],
        name=scm_type,
        marker_color=color,
        line_color=color,
        fillcolor=color,
        boxpoints=False,
        width=box_width * 0.6,
        marker=dict(size=5, opacity=0.6, line=dict(width=0.5, color='white')),
        customdata=subset[['Mix Label']].values,
        hovertemplate='<b>%{customdata[0]}</b><br>GWP: %{y} kg CO₂e/cy<extra>' + scm_type + '</extra>',
    ))

# --- Scatter dot traces ---
for scm_type, color in zip(scm_plot_order, colors):
    subset    = df_plot[df_plot['scm_type'] == scm_type]
    is_no_scm = (scm_type == 'No SCM')
    sampled_x, sampled_y, sampled_names = [], [], []

    for strength in strength_order:
        bucket = subset[subset['Compressive_Strength'] == strength]
        n = min(MAX_DOTS_PER_BUCKET, len(bucket))
        if n == 0:
            continue
        sample = bucket.sample(n=n, random_state=42) if n < len(bucket) else bucket
        x_center = strength_to_idx[strength] + offsets_map[scm_type]
        jitter = rng.uniform(-POINT_JITTER_WIDTH, POINT_JITTER_WIDTH, n)
        sampled_x.extend(x_center + jitter)
        sampled_y.extend(sample['gwp_rounded'].values)
        sampled_names.extend(sample['Mix Label'].values)

    opacity = 0.35 if is_no_scm else 0.6
    fig.add_trace(go.Scatter(
        x=sampled_x,
        y=sampled_y,
        mode='markers',
        marker=dict(color=color, size=5, opacity=opacity,
                    line=dict(width=0.5, color='white')),
        customdata=[[n] for n in sampled_names],
        hovertemplate='<b>%{customdata[0]}</b><br>GWP: %{y} kg CO₂e/cy<extra>' + scm_type + '</extra>',
        showlegend=False,
        name=scm_type + ' (dots)',
    ))

# --- Count annotations ---
for scm_type in scm_plot_order:
    for strength in strength_order:
        subset = df_plot[
            (df_plot['scm_type'] == scm_type) &
            (df_plot['Compressive_Strength'] == strength)
        ]
        if len(subset) == 0:
            continue

        count = len(subset)
        vals = subset['gwp_rounded']
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        whisker_top = min(q3 + 1.5 * (q3 - q1), vals.max())

        fig.add_annotation(
            x=strength_to_idx[strength] + offsets_map[scm_type],
            y=whisker_top,
            text=str(count),
            textangle=-90,
            showarrow=False,
            yanchor='bottom',
            font=dict(size=9, color='#444444'),
            xref='x', yref='y',
        )

# --- Footnotes ---
fig.add_annotation(
    text=(
        'Numbers at tops of whiskers denote count of concrete EPDs<br>'
        'Random sampling of dots has been removed for visual clarity where applicable'
    ),
    xref='paper', yref='paper',
    x=1.0, y=-0.13,
    xanchor='right', yanchor='top',
    showarrow=False,
    font=dict(size=10, color='#888888'),
    align='right',
)

# --- Vertical separators ---
y_min = df_plot['gwp_rounded'].min()
y_max = df_plot['gwp_rounded'].max()

for i in range(len(strength_order) - 1):
    fig.add_shape(
        type='line',
        x0=i + 0.5, y0=y_min - 20,
        x1=i + 0.5, y1=y_max + 60,
        line=dict(color='rgba(128,128,128,0.3)', width=1, dash='dash'),
        layer='below'
    )

# --- Axes ---
fig.update_xaxes(
    tickvals=list(range(len(strength_order))),
    ticktext=[f'{int(s)} psi' for s in strength_order],
    tickangle=-45,
    title_text='Compressive Strength',
    range=[-0.5, len(strength_order) - 0.5],
)

fig.update_yaxes(
    title_text='GWP (kg CO₂e per cubic yard)',
    showgrid=True,
    gridcolor='lightgrey',
    range=[y_min - 20, y_max + 80],
)

fig.update_layout(
    title={
        'text': 'GWP Distribution by Compressive Strength and SCM Type',
        'x': 0.5, 'xanchor': 'center', 'yanchor': 'top',
        'font': {'size': 16, 'family': 'Arial', 'color': 'black'},
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    width=1400,
    margin=dict(b=120),
    legend={
        'title': 'SCM Type',
        'orientation': 'v',
        'yanchor': 'top', 'y': 0.99,
        'xanchor': 'left', 'x': 1.01,
    },
)

print(f"Total samples (2500–8000 psi): {len(df_filtered)}")
print(f"\nSamples by SCM Type:")
print(df_filtered['scm_type'].value_counts())

print(f"\nDots shown per bucket (capped at {MAX_DOTS_PER_BUCKET}):")
for scm_type in scm_plot_order:
    scm_subset = df_plot[df_plot['scm_type'] == scm_type]
    sampled = sum(min(MAX_DOTS_PER_BUCKET, len(scm_subset[scm_subset['Compressive_Strength'] == s])) for s in strength_order)
    total   = len(scm_subset)
    print(f"  {scm_type}: {sampled:,} shown / {total:,} total")

fig.show()

Total samples (2500–8000 psi): 43110

Samples by SCM Type:
scm_type
Fly Ash                17572
No SCM                 14068
Slag                    8756
Both Fly Ash & Slag     2714
Name: count, dtype: int64

Dots shown per bucket (capped at 600):
  No SCM: 4,478 shown / 14,068 total
  Fly Ash: 4,865 shown / 17,572 total
  Slag: 4,735 shown / 8,756 total
  Both Fly Ash & Slag: 2,537 shown / 2,714 total


In [10]:
fig_pie = make_subplots(
    rows=1,
    cols=len(strength_order),
    specs=[[{'type': 'pie'}] * len(strength_order)],
    subplot_titles=[f'{int(s)} psi' for s in strength_order],
    horizontal_spacing=0.005,
)

for col_idx, strength in enumerate(strength_order, start=1):
    bucket = df_plot[df_plot['Compressive_Strength'] == strength]
    counts = bucket['scm_type'].value_counts()

    labels, values, pie_colors = [], [], []
    for scm_type, color in zip(scm_plot_order, colors):
        count = counts.get(scm_type, 0)
        if count > 0:
            labels.append(scm_type)
            values.append(count)
            pie_colors.append(color)

    total = sum(values)
    pct_texts = [f'{round(v / total * 100)}%' for v in values]

    fig_pie.add_trace(
        go.Pie(
            labels=labels,
            values=values,
            marker=dict(colors=pie_colors),
            text=pct_texts,
            textinfo='text',
            hovertemplate='%{label}<br>%{text} of %{value:,} mixes<extra></extra>',
            showlegend=False,
            textfont=dict(size=10),
            insidetextorientation='auto',
        ),
        row=1, col=col_idx,
    )

fig_pie.update_layout(
    title={
        'text': 'Percentages of EPDs by SCM Type within Each Strength Bucket',
        'x': 0.5, 'xanchor': 'center', 'yanchor': 'top',
        'font': {'size': 16, 'family': 'Arial', 'color': 'black'},
    },
    height=300,
    width=1400,
    paper_bgcolor='white',
    margin=dict(t=110, b=20, l=10, r=10),
)

fig_pie.show()

In [11]:
# EPD counts and percentages by SCM type
total = len(df_plot)
print(f"Total EPDs (2500–8000 psi): {total:,}\n")
print(f"{'SCM Type':<25} {'Count':>8} {'Percent':>9}")
print("-" * 45)
for scm_type in scm_plot_order:
    count = (df_plot['scm_type'] == scm_type).sum()
    pct = count / total * 100
    print(f"{scm_type:<25} {count:>8,} {pct:>8.1f}%")

Total EPDs (2500–8000 psi): 43,110

SCM Type                     Count   Percent
---------------------------------------------
No SCM                      14,068     32.6%
Fly Ash                     17,572     40.8%
Slag                         8,756     20.3%
Both Fly Ash & Slag          2,714      6.3%


### Save Plots

In [12]:
from kaleido import Kaleido

async with Kaleido(n=4) as k:
    await k.write_fig(fig, path='../tests/broyles_gwp_by_compressive_strength_scm.png',
                      opts={'format': 'png', 'scale': 3, 'width': fig.layout.width, 'height': fig.layout.height})
    await k.write_fig(fig_pie, path='../tests/broyles_scm_type_breakdown_pies.png',
                      opts={'format': 'png', 'scale': 3, 'width': fig_pie.layout.width, 'height': fig_pie.layout.height})
print("Saved PNGs to tests/")

Saved PNGs to tests/
